# 如果不使用langchain 该如何创建Tool和Agent

脱离 LangChain，直接使用 OpenAI 原生 API 构建 Agent 其实非常有助于理解大模型底层是如何工作的。

核心机制只有两个：

- Tools (工具)：就是普通的 Python 函数 + 一个描述这个函数的 JSON (Schema)。
- Agent (智能体)：一个 while 循环，负责把函数定义发给 GPT，如果 GPT 说“我要调用函数”，代码就去执行函数，把结果再塞回给 GPT。

下面是纯原生 Python + OpenAI 的实现代码。

## 前置准备

你需要安装官方库。

pip install openai duckduckgo-search

## 完整代码实现

这是一个完整的、不依赖 LangChain 的 ReAct Agent 实现。

In [2]:
import os
import json
import dotenv
from openai import OpenAI
from duckduckgo_search import DDGS

# 1. 加载配置
dotenv.load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

# ==========================================
# 第一步：定义实际执行的 Python 函数 (Tool Body)
# ==========================================

def search_web(query):
    """实际执行搜索的函数"""
    print(f"🔍 [系统正在执行本地代码] 搜索关键词: {query}")
    try:
        results = []
        ddgs = DDGS()
        # 搜索最近一天的中文结果
        ddg_gen = ddgs.text(query, region="cn-zh", timelimit="d", max_results=3)
        if ddg_gen:
            for r in ddg_gen:
                results.append(f"标题: {r['title']}\n内容: {r['body']}")
        return "\n---\n".join(results) if results else "未找到结果"
    except Exception as e:
        return f"搜索出错: {str(e)}"

# ==========================================
# 第二步：定义工具的描述 (Tool Schema)
# ==========================================
# 这是 OpenAI 规定的格式，必须告诉模型函数名、描述、参数类型
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_web", # 必须与上面的函数名对应
            "description": "用于搜索互联网实时信息，如天气、新闻、股票等。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "需要搜索的具体关键词，例如 '北京天气'"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# 创建一个映射表，方便后续根据名字调用函数
available_functions = {
    "search_web": search_web,
}

# ==========================================
# 第三步：构建 Agent (执行循环)
# ==========================================

def run_agent(user_input):
    # 1. 初始化对话历史
    messages = [
        {"role": "system", "content": "你是一个有用的助手。如果不知道答案，请使用搜索工具。"},
        {"role": "user", "content": user_input}
    ]

    print(f"👤 用户: {user_input}")

    # 2. 进入思考-执行循环 (ReAct Loop)
    while True:
        # 向 GPT 发送请求，带上 tools 定义
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools_schema, # 关键：告诉模型有哪些工具可用
            tool_choice="auto"  # 让模型自己决定是否用工具
        )

        response_message = response.choices[0].message

        # 检查模型是否想调用工具
        tool_calls = response_message.tool_calls

        if tool_calls:
            # === 情况 A: 模型决定调用工具 ===

            # 必须把模型的这个“决定”加入历史，否则下次对话会报错
            messages.append(response_message)

            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                # 从映射表中找到对应的 Python 函数
                function_to_call = available_functions.get(function_name)

                if function_to_call:
                    # 执行 Python 函数
                    function_response = function_to_call(
                        query=function_args.get("query")
                    )

                    # 将函数执行结果封装成 tool 类型的消息存入历史
                    messages.append(
                        {
                            "tool_call_id": tool_call.id, # 必须带上 ID，让模型知道这是哪个调用的结果
                            "role": "tool",
                            "name": function_name,
                            "content": function_response,
                        }
                    )

            # 注意：这里不 return，而是 continue，让循环继续，
            # 把“函数执行结果”发回给模型，让模型根据结果生成最终答案。
            print("🤖 模型正在思考工具返回的结果...")

        else:
            # === 情况 B: 模型没有调用工具，直接返回了回答 ===
            print(f"✅ AI 回复: {response_message.content}")
            return response_message.content

# ==========================================
# 运行
# ==========================================
if __name__ == "__main__":
    run_agent("你知道特斯拉Model Y汽车吗？")

👤 用户: 你知道特斯拉Model Y汽车吗？
✅ AI 回复: 是的，特斯拉Model Y是一款由特斯拉公司生产的电动SUV。它是在2019年首次发布的，并且是特斯拉Model 3的衍生车型。Model Y的设计注重空间和实用性，提供了较大的内部空间和可选的第三排座位。

一些Model Y的主要特点包括：

1. **电动驱动**：Model Y使用电动驱动系统，提供瞬时加速和高效能。
2. **续航里程**：根据不同的配置和电池容量，Model Y的续航里程可达300英里以上（约480公里）。
3. **内部空间**：与Model 3相比，Model Y提供更多的货物空间和舒适性，内部设计简约现代。
4. **安全性**：特斯拉的电动车通常在安全性测试中表现优异，Model Y也不例外。
5. **自动驾驶功能**：Model Y可配备特斯拉的自动驾驶功能，包括部分自动驾驶和全自动驾驶选项。

如果你有更具体的问题或想了解更多信息，请告诉我！


## 原理拆解：不使用 LangChain 做了什么？

1. Tool 定义 (Schema):
    - 在 LangChain 中，StructuredTool 帮你自动生成了 JSON Schema。
    - 在这里，我们手动写了一个 tools_schema 字典。这就是 OpenAI API 标准的 tools 参数格式。
2. Tool 执行 (Dispatch):
    - 在 LangChain 中，AgentExecutor 帮你去匹配函数名并执行。
    - 在这里，我们创建了一个 available_functions 字典，手动解析 GPT 返回的 JSON，找到对应的 Python 函数并运行。
3. 消息历史管理:
    - 在 LangChain 中，它自动处理 HumanMessage, AIMessage, ToolMessage 的堆叠。
    - 在这里，我们需要手动 messages.append(...)，特别是必须严格遵守 OpenAI 的格式要求（比如 tool_call_id 必须对应）。


## 优缺点对比

- 原生方式 (本代码):
    - 优点: 极其透明，你完全知道发生了什么；调试极其容易；不依赖臃肿的第三方库；代码量其实也不大。
    - 缺点: 手写 JSON Schema 比较麻烦（容易写错）；手动管理对话历史稍微繁琐。
- LangChain:
    - 优点: 快速集成，一行代码把 Python 函数转成 Tool；自动管理记忆和回调。
    - 缺点: 封装太深，出 Bug 了很难排查；版本更新太快导致代码容易过时。